## 성능 비교 확인 위한 이미지 선택

In [ ]:
# ==============================================================================
# 1. 데이터 저장 폴더 생성
# ==============================================================================
!mkdir -p /content/DIV2K

# 3. Valid HR 데이터 다운로드 및 압축 해제
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip -O /content/DIV2K/valid_hr.zip
!unzip -q /content/DIV2K/valid_hr.zip -d /content/DIV2K
!rm /content/DIV2K/valid_hr.zip

print("데이터셋 다운로드 완료")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, glob, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# ---- 해당 경로에 best_model.path 저장된 후에 사용 가능 ----
EXP_DIRS = {
    "baseline":  "/content/drive/MyDrive/SRNO_Base_final",
    "fourier5":  "/content/drive/MyDrive/SRNO_Fourier5_final",
    "fourier10": "/content/drive/MyDrive/SRNO_Fourier10_final",
    "inv_area":  "/content/drive/MyDrive/SRNO_LocalAgg_final",
}

# ---- DIV2K valid HR 경로 ----
DIV2K_ROOT = "/content/DIV2K"
VALID_GLOB = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR", "*.png")
valid_files = sorted(glob.glob(VALID_GLOB))
print("Valid images:", len(valid_files))
assert len(valid_files) > 0, "DIV2K_valid_HR 경로가 비었어. DIV2K 다운로드/경로 확인!"

In [ ]:
def norm_01_to_m11(x):
    return (x - 0.5) / 0.5

def denorm_m11_to_01(x):
    return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)

    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1,2)
    if batch_size != 1:
        coords = coords.repeat(batch_size, 1, 1)
        cell = cell.repeat(batch_size, 1)
    return coords, cell

def rgb_to_y(img):  # img (...,3) in [0,1]
    return 0.2567 * img[...,0] + 0.5041 * img[...,1] + 0.0979 * img[...,2] + 16/255

@torch.no_grad()
def calc_psnr_y(pred01, target01, shave=2):
    pred01 = pred01.clamp(0,1)
    target01 = target01.clamp(0,1)

    pred01 = pred01[..., shave:-shave, shave:-shave]
    target01 = target01[..., shave:-shave, shave:-shave]

    pred = pred01.permute(0,2,3,1)
    tgt  = target01.permute(0,2,3,1)

    y_pred = rgb_to_y(pred)
    y_tgt  = rgb_to_y(tgt)

    mse = torch.mean((y_pred - y_tgt) ** 2).item()
    if not math.isfinite(mse) or mse <= 0:
        return float("nan") if not math.isfinite(mse) else 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))

In [ ]:
class FourierPositionalEncoding(nn.Module):
    def __init__(self, L=5):
        super().__init__()
        self.L = L
        self.register_buffer("freq_bands", 2 ** torch.linspace(0, L-1, L), persistent=False)

    def forward(self, x):  # (B,N,2)
        pe = []
        for freq in self.freq_bands.to(x.device):
            pe.append(torch.sin(x * freq * math.pi))
            pe.append(torch.cos(x * freq * math.pi))
        return torch.cat(pe, dim=-1)  # (B,N, 2*L*2)

class GalerkinAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        # 논문 Eq.12: LN은 K,V에만
        self.ln_k = nn.LayerNorm(dim)
        self.ln_v = nn.LayerNorm(dim)

        self.to_out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        q = self.to_q(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)
        k = self.to_k(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)
        v = self.to_v(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)

        # LN은 (B,N,C)에서 적용 (token별 feature norm)
        k2 = k.permute(0, 2, 1, 3).reshape(B, N, C)  # (B,N,C)
        v2 = v.permute(0, 2, 1, 3).reshape(B, N, C)  # (B,N,C)

        k2 = self.ln_k(k2)
        v2 = self.ln_v(v2)

        k = k2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)
        v = v2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)

        # Galerkin: Q (K^T V) / N
        context = torch.matmul(k.transpose(-2, -1), v) / float(N)  # (B,H,D,D)
        out = torch.matmul(q, context)                             # (B,H,N,D)

        out = out.permute(0, 2, 1, 3).reshape(B, N, C)
        return self.to_out(out)

class ResBlock(nn.Module):
    def __init__(self, n_feats, kernel_size=3, act=nn.ReLU(True), res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
            act,
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR(nn.Module):
    def __init__(self, n_resblocks=16, n_feats=64):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(3, n_feats, 3, padding=1)
        )
        m_body = [ResBlock(n_feats, 3, res_scale=0.1) for _ in range(n_resblocks)]
        m_body.append(nn.Conv2d(n_feats, n_feats, 3, padding=1))
        self.body = nn.Sequential(*m_body)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        return x + res

class SRNO_Fourier(nn.Module):
    '''
    SRNO-like implicit SR model.

    - Baseline (SRNO-like): 4 corners를 그대로 concat해서 lifting에 넣음
      corner_agg="concat"

    - 실험(B): 4 corners를 inverse-area weighting으로 1개로 압축 (LIIF-style local ensemble)
      corner_agg="inv_area"
    '''
    def __init__(self, residual_scale=0.1, use_fourier=False, L=5, width=128, blocks=8, corner_agg="concat"):
        super().__init__()
        assert corner_agg in ["concat", "inv_area"], "corner_agg must be 'concat' or 'inv_area'"
        self.corner_agg = corner_agg

        self.encoder = EDSR(16, 64)
        self.use_fourier = use_fourier
        self.L = L
        self.pos_enc = FourierPositionalEncoding(L=L)

        # corner feature dim = q_feat(64) + rel_coord(2) + (fourier) + rel_cell(2)
        corner_dim = 64 + 2 + 2
        if use_fourier:
            corner_dim += (2 * L * 2)

        in_dim = (4 * corner_dim) if (self.corner_agg == "concat") else corner_dim

        self.latent_dim = width
        self.lifting = nn.Linear(in_dim, self.latent_dim)

        self.layers = nn.ModuleList([GalerkinAttention(self.latent_dim, heads=8) for _ in range(blocks)])
        self.ffns = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(self.latent_dim),
                nn.Linear(self.latent_dim, self.latent_dim),
                nn.GELU(),
                nn.Linear(self.latent_dim, self.latent_dim),
            ) for _ in range(blocks)
        ])

        self.projection = nn.Linear(self.latent_dim, 3)
        self.residual_scale = residual_scale

    @staticmethod
    def _feat_coord_grid(Hf, Wf, device, B):
        y = torch.linspace(-1, 1, Hf, device=device, dtype=torch.float32)
        x = torch.linspace(-1, 1, Wf, device=device, dtype=torch.float32)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        grid = torch.stack([xx, yy], dim=-1).unsqueeze(0)  # (1,Hf,Wf,2)
        return grid.repeat(B, 1, 1, 1)  # (B,Hf,Wf,2)

    def forward(self, x, coords, cell):
        B = x.shape[0]

        coords = coords.to(dtype=torch.float32)
        cell   = cell.to(dtype=torch.float32)

        grid = coords.unsqueeze(2)  # (B,N,1,2)

        # base from LR (bilinear)
        base = F.grid_sample(x, grid, mode="bilinear", align_corners=False)  # (B,3,N,1)
        base = base.squeeze(3).permute(0,2,1)  # (B,N,3)

        feat = self.encoder(x)  # (B,64,Hf,Wf)
        _, _, Hf, Wf = feat.shape

        feat_coord = self._feat_coord_grid(Hf, Wf, feat.device, B).permute(0,3,1,2)  # (B,2,Hf,Wf)

        rx, ry = 1.0 / Hf, 1.0 / Wf
        eps = 1e-6
        vx_lst = [-1, 1]
        vy_lst = [-1, 1]

        rel_cell = cell.unsqueeze(1).repeat(1, coords.shape[1], 1).clone()
        rel_cell[..., 0] *= Hf
        rel_cell[..., 1] *= Wf

        preds = []
        areas = []
        for vx in vx_lst:
            for vy in vy_lst:
                coord_ = coords.clone()
                coord_[..., 0] = (coord_[..., 0] + vx * rx).clamp(-1 + eps, 1 - eps)
                coord_[..., 1] = (coord_[..., 1] + vy * ry).clamp(-1 + eps, 1 - eps)
                grid_ = coord_.unsqueeze(2)

                q_feat = F.grid_sample(feat, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)
                q_coord = F.grid_sample(feat_coord, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)

                rel_coord = coords - q_coord
                rel_coord[..., 0] *= Hf
                rel_coord[..., 1] *= Wf

                # inverse-area weighting에서 쓰는 area proxy
                area = (rel_coord[..., 0].abs() * rel_coord[..., 1].abs())  # (B,N)
                areas.append(area)

                parts = [q_feat, rel_coord]
                if self.use_fourier:
                    parts.append(self.pos_enc(rel_coord))
                parts.append(rel_cell)

                preds.append(torch.cat(parts, dim=-1))  # (B,N,corner_dim)

        if self.corner_agg == "concat":
            z_in = torch.cat(preds, dim=-1)  # (B,N,4*corner_dim)
        else:
            eps_w = 1e-9
            inv = [1.0 / (a + eps_w) for a in areas]
            inv_sum = (inv[0] + inv[1] + inv[2] + inv[3]).clamp_min(eps_w)
            w = [(iv / inv_sum).unsqueeze(-1) for iv in inv]
            z_in = preds[0]*w[0] + preds[1]*w[1] + preds[2]*w[2] + preds[3]*w[3]  # (B,N,corner_dim)

        z = self.lifting(z_in)

        for attn, ffn in zip(self.layers, self.ffns):
            z = z + attn(z)
            z = z + ffn(z)

        residual = self.projection(z) * self.residual_scale
        return base + residual

In [ ]:
def build_model_from_config(cfg):
    return SRNO_Fourier(
        residual_scale=cfg.get("RESIDUAL_SCALE", 0.1),
        use_fourier=cfg.get("USE_FOURIER", False),
        L=cfg.get("L", 5),
        width=cfg.get("WIDTH", 128),
        blocks=cfg.get("BLOCKS", 8),
        corner_agg=cfg.get("CORNER_AGG", "concat"),
    ).to(DEVICE)

def load_best_model(exp_dir, *, prefer_ckpt=True, override_cfg=None):
    ckpt_path = os.path.join(exp_dir, "best_checkpoint.pth")
    sd_path   = os.path.join(exp_dir, "best_model.pth")

    if prefer_ckpt and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        cfg = dict(ckpt.get("config", {}))
        if override_cfg:
            cfg.update(override_cfg)
        model = build_model_from_config(cfg)
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        model.eval()
        return model, cfg

    if not os.path.exists(sd_path):
        raise FileNotFoundError(f"Neither {ckpt_path} nor {sd_path} exists in {exp_dir}")

    if not override_cfg:
        raise ValueError(
            "best_checkpoint.pth가 없고 best_model.pth만 존재함. "
            "best_model.pth에는 config가 없어서 override_cfg를 줘야 로드 가능."
        )

    sd = torch.load(sd_path, map_location="cpu")
    model = build_model_from_config(override_cfg)
    model.load_state_dict(sd, strict=True)
    model.eval()
    return model, override_cfg

# ---- 4개 모델 로드 ----
models = {}
model_cfgs = {}
for name, d in EXP_DIRS.items():
    models[name], model_cfgs[name] = load_best_model(d, prefer_ckpt=True)

print("✅ models loaded:", list(models.keys()))
for k in ["baseline","fourier5","fourier10","inv_area"]:
    if k in model_cfgs:
        cfg = model_cfgs[k]
        print(f"  - {k}: USE_FOURIER={cfg.get('USE_FOURIER')} L={cfg.get('L')} "
              f"CORNER_AGG={cfg.get('CORNER_AGG')} WIDTH={cfg.get('WIDTH')} BLOCKS={cfg.get('BLOCKS')}")

In [ ]:
def get_crop_transform(hr_size):
    return transforms.Compose([transforms.CenterCrop(hr_size), transforms.ToTensor()])

@torch.no_grad()
def run_one_image(path, *, hr_size=96, scale=2.0):
    """
    hr_size: GT 패치 크기 (예: 96, 128, 192)
    scale: 2.0(학습과 동일), 3.0/4.0(OOD 테스트)
    """
    tfm = get_crop_transform(hr_size)
    img = Image.open(path).convert("RGB")
    hr = tfm(img).unsqueeze(0).to(DEVICE)  # (1,3,hr,hr) in [0,1]

    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)
    lr = F.interpolate(hr, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    hr_n = norm_01_to_m11(hr)
    lr_n = norm_01_to_m11(lr)

    # ----------------------------------------
    # ✅ 학습 코드와 동일한 scale-conditioned cell
    # ----------------------------------------
    coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)

    scale_tensor = torch.tensor(scale, device=DEVICE, dtype=torch.float32)
    cell = cell * scale_tensor.view(1, 1)   # 🔥 핵심 추가

    # ----------------------------------------

    outs = {}
    psnrs = {}
    for name, m in models.items():
        pred_n = m(lr_n, coords, cell)  # (1, N, 3)
        pred_n = pred_n.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()
        pred01 = denorm_m11_to_01(pred_n)

        outs[name] = pred01
        psnrs[name] = calc_psnr_y(pred01, hr, shave=2)

    lr_up = F.interpolate(lr, size=(hr_size, hr_size), mode="bicubic", align_corners=False).clamp(0,1)

    return lr_up, hr.clamp(0,1), outs, psnrs

def show_grid_pretty(lr_up, hr, outs, psnrs, title="", save_path=None):
    order = [
        ("baseline", "Base"),
        ("fourier5", "Fourier (L=5)"),
        ("fourier10", "Fourier (L=10)"),
        ("inv_area", "Local Ensemble (inv-area)")
    ]

    fig, axes = plt.subplots(1, 2 + len(order), figsize=(4*(2+len(order)), 4))

    def to_np(t):
        return t.squeeze(0).permute(1,2,0).detach().cpu().numpy().clip(0,1)

    axes[0].imshow(to_np(lr_up))
    axes[0].set_title("LR (bicubic up)")
    axes[0].axis("off")

    axes[1].imshow(to_np(hr))
    axes[1].set_title("GT")
    axes[1].axis("off")

    for i, (k, name) in enumerate(order):
        img = to_np(outs[k])
        axes[2+i].imshow(img)
        axes[2+i].set_title(f"{name}\n(PSNR(Y)={psnrs[k]:.2f})")
        axes[2+i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=220, bbox_inches="tight")

    plt.show()

def random_crop_xy(img_w, img_h, crop):
    if img_w <= crop or img_h <= crop:
        return 0, 0
    x = random.randint(0, img_w - crop)
    y = random.randint(0, img_h - crop)
    return x, y

def crop_pil(img, x, y, crop):
    return img.crop((x, y, x + crop, y + crop))

@torch.no_grad()
def high_freq_score(img01):
    """img01: (1,3,H,W) in [0,1]"""
    gray = 0.299*img01[:,0:1] + 0.587*img01[:,1:2] + 0.114*img01[:,2:3]
    k = torch.tensor([[[[0,-1,0],[-1,4,-1],[0,-1,0]]]], device=gray.device, dtype=gray.dtype)
    lap = F.conv2d(gray, weight=k, padding=1)
    return torch.mean(lap**2).item()

In [ ]:
# ============================================================
# 1) Core eval: "evaluate a given HR patch tensor"
#    (모델 4개 forward + PSNR 계산)
# ============================================================

@torch.no_grad()
def eval_patch(hr01, *, scale=2.0):
    """
    hr01: (1,3,HR,HR) in [0,1]
    return: lr_up01, outs01(dict), psnr(dict)
    """
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    hr_n = norm_01_to_m11(hr01)
    lr_n = norm_01_to_m11(lr01)

    coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)
    cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)

    outs, psnrs = {}, {}
    for name, m in models.items():
        pred_n = m(lr_n, coords, cell)  # (1,N,3)
        pred_n = pred_n.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()
        pred01 = denorm_m11_to_01(pred_n).clamp(0,1)

        outs[name] = pred01
        psnrs[name] = calc_psnr_y(pred01, hr01, shave=2)

    lr_up01 = F.interpolate(lr01, size=(hr_size, hr_size), mode="bicubic", align_corners=False).clamp(0,1)
    return lr_up01, outs, psnrs

In [ ]:
# ============================================================
# 2) Best-patch finder per image (랜덤 n_trials 중 hf max)
# ============================================================

@torch.no_grad()
def find_best_patch(path, *, hr_size=96, n_trials=10):
    """
    return: (hr01, hf_best, x, y)
    """
    img = Image.open(path).convert("RGB")
    W, H = img.size

    best = None  # (hf, hr01, x, y)
    for _ in range(n_trials):
        x, y = random_crop_xy(W, H, hr_size)
        hr_pil = crop_pil(img, x, y, hr_size)
        hr01 = transforms.ToTensor()(hr_pil).unsqueeze(0).to(DEVICE)

        hf = high_freq_score(hr01)
        if (best is None) or (hf > best[0]):
            best = (hf, hr01, x, y)

    hf_best, hr01, x, y = best
    return hr01, hf_best, x, y

In [ ]:
# ============================================================
# 3) Score all valid with bestpatch + PSNR filter
# ============================================================

@torch.no_grad()
def edge_density(img01, thr=0.08):
    """
    img01: (1,3,H,W) in [0,1]
    return: edge pixel ratio (0~1)
    """
    gray = 0.299*img01[:,0:1] + 0.587*img01[:,1:2] + 0.114*img01[:,2:3]

    kx = torch.tensor([[[[-1,0,1],[-2,0,2],[-1,0,1]]]], device=gray.device, dtype=gray.dtype) / 8.0
    ky = torch.tensor([[[[-1,-2,-1],[0,0,0],[1,2,1]]]], device=gray.device, dtype=gray.dtype) / 8.0

    gx = F.conv2d(gray, kx, padding=1)
    gy = F.conv2d(gray, ky, padding=1)
    g  = torch.sqrt(gx*gx + gy*gy)

    return (g > thr).float().mean().item()

@torch.no_grad()
def orient_structure_score(hr01, thr=0.08, angle_deg=15.0):
    """Axis-aligned structure score in [0,1]. Higher => more horizontal/vertical edges (text/buildings)."""
    # hr01: (1,3,H,W) in [0,1]
    gray = 0.299*hr01[:,0:1] + 0.587*hr01[:,1:2] + 0.114*hr01[:,2:3]

    kx = torch.tensor([[[[-1,0,1],[-2,0,2],[-1,0,1]]]], device=gray.device, dtype=gray.dtype) / 8.0
    ky = torch.tensor([[[[-1,-2,-1],[0,0,0],[1,2,1]]]], device=gray.device, dtype=gray.dtype) / 8.0

    gx = F.conv2d(gray, kx, padding=1)
    gy = F.conv2d(gray, ky, padding=1)
    mag = torch.sqrt(gx*gx + gy*gy)

    m = (mag > thr)
    if m.float().mean().item() < 1e-6:
        return 0.0

    ang = torch.atan2(gy, gx)  # [-pi, pi]

    def ang_dist(a, target):
        # circular distance in radians
        d = torch.abs((a - target + math.pi) % (2*math.pi) - math.pi)
        return d

    tol = float(angle_deg) * math.pi / 180.0
    d0  = torch.minimum(ang_dist(ang, 0.0), ang_dist(ang, math.pi))
    d90 = ang_dist(ang, math.pi/2)

    axis = ((d0 < tol) | (d90 < tol)) & m
    return axis.float().mean().item()


@torch.no_grad()
def collect_candidate_patches(path, *, hr_size=192, n_trials=30, topk=5, edge_thr=0.08):
    """Collect top-k candidate patches by (edge_density + axis_structure)."""
    img = Image.open(path).convert("RGB")
    W, H = img.size

    cand = []
    for _ in range(int(n_trials)):
        x, y = random_crop_xy(W, H, hr_size)
        hr_pil = crop_pil(img, x, y, hr_size)
        hr01 = transforms.ToTensor()(hr_pil).unsqueeze(0).to(DEVICE)

        # quick reject: too flat
        if hr01.var().item() < 1e-4:
            continue

        hf = high_freq_score(hr01)
        ed = edge_density(hr01, thr=edge_thr)
        oscore = orient_structure_score(hr01, thr=edge_thr)

        cand.append({"path": path, "hr01": hr01, "x": x, "y": y, "hf": hf, "ed": ed, "os": oscore})
    # 1차: 구조/엣지 기반 topk 남기기
    if len(cand) == 0:
        return []   # ✅ None 말고 빈 리스트

    cand = sorted(cand, key=lambda r: (r["ed"]*0.6 + r["os"]*0.4), reverse=True)[:topk]
    return cand

@torch.no_grad()
def pick_best_visual_patch(
    path, *,
    scale=2.0, hr_size=192,
    n_trials=30, topk=5,
    edge_thr=0.08,
    min_psnr=0.0, max_psnr=1e9,
    min_edge=0.0, min_hf=None
):
    """Pick ONE patch that is (1) visually structured (2) models disagree a lot (gap)."""
    cand = collect_candidate_patches(
        path,
        hr_size=hr_size,
        n_trials=n_trials,
        topk=topk,
        edge_thr=edge_thr
    )
    if not cand:
        return None
    best = None
    for r in cand:
        hr01 = r["hr01"]
        hf, ed, oscore = r["hf"], r["ed"], r["os"]

        if ed < float(min_edge):
            continue
        if (min_hf is not None) and (hf < float(min_hf)):
            continue

        lr_up01, outs, ps = eval_patch(hr01, scale=scale)

        # psnr filter
        ref_val = ps.get("fourier10", None)
        if ref_val is not None:
            if ref_val < float(min_psnr) or ref_val > float(max_psnr):
                continue

        vals = list(ps.values())
        gap = max(vals) - min(vals)
        improve = ps["fourier10"] - ps["baseline"]

        # 최종 점수: disagreement + structure
        score = 0.35*gap + 0.25*ed + 0.25*oscore + 0.15*hf


        if (best is None) or (score > best["score"]):
            best = {
                "path": path,
                "x": r["x"], "y": r["y"], "hr_size": hr_size,
                "hf": hf, "ed": ed, "os": oscore,
                "psnr": ps,
                "gap": gap,
                "improve": improve,
                "score": score,
            }
    return best


@torch.no_grad()
def score_all_valid_visual(
    *, hr_size=192, scale=2.0,
    n_trials=30, topk=5,
    min_psnr=30.0, max_psnr=45.0,
    min_edge=0.03, edge_thr=0.08,
    min_hf=None
):
    rows = []
    for p in tqdm(valid_files, desc=f"visual_pick x{scale}"):
        best = pick_best_visual_patch(
            p,
            scale=scale, hr_size=hr_size,
            n_trials=n_trials, topk=topk,
            edge_thr=edge_thr,
            min_psnr=min_psnr, max_psnr=max_psnr,
            min_edge=min_edge, min_hf=min_hf
        )
        if best is None:
            continue
        rows.append(best)
    return rows

@torch.no_grad()
def score_all_valid_bestpatch(
    *, hr_size=96, scale=2.0, n_trials=10,
    min_psnr=30.0, max_psnr=45.0,
    min_edge=0.03, edge_thr=0.08,
    min_hf=None,
    psnr_ref="fourier10"
):
    rows = []
    for p in tqdm(valid_files, desc=f"valid(bestpatch) hr={hr_size} scale={scale} trials={n_trials} PSNR[{min_psnr},{max_psnr}] edge>={min_edge}"):
        hr01, hf, x, y = find_best_patch(p, hr_size=hr_size, n_trials=n_trials)
        lr_up01, outs, ps = eval_patch(hr01, scale=scale)

        ref = float(ps.get(psnr_ref, -1e9))

        # PSNR 하한 + 상한
        if ref < float(min_psnr) or ref > float(max_psnr):
            continue

        # edge density (윤곽/선 많은 패치만)
        ed = edge_density(hr01, thr=edge_thr)
        if ed < float(min_edge):
            continue

        # (선택) hf 최소치
        if (min_hf is not None) and (hf < float(min_hf)):
            continue

        vals = list(ps.values())
        gap = max(vals) - min(vals)
        improve = ps["fourier10"] - ps["baseline"]

        rows.append({
            "path": p,
            "x": x, "y": y, "hr_size": hr_size,
            "hf": hf,
            "ed": ed,
            "psnr": ps,
            "gap": gap,
            "improve": improve,
        })
    return rows

In [ ]:
from tqdm import tqdm
import numpy as np

@torch.no_grad()
def cheap_rank_one_image(path, hr_size=160, n_trials=15, edge_thr=0.08):
    """
    모델 forward 없이: hf/edge/structure로 '이 이미지에서 뽑히는 패치들이 얼마나 볼만한가'를 점수화
    반환:
      - best_score: 해당 이미지에서 관측된 최고 cheap score
      - best_meta: best 패치의 (x,y,hf,ed,os) 메타
    """
    img = Image.open(path).convert("RGB")
    W, H = img.size

    best_score = -1e9
    best_meta = None

    for _ in range(n_trials):
        x, y = random_crop_xy(W, H, hr_size)
        hr_pil = crop_pil(img, x, y, hr_size)
        hr01 = transforms.ToTensor()(hr_pil).unsqueeze(0).to(DEVICE)

        # 평평한 패치 제거
        if hr01.var().item() < 1e-4:
            continue

        hf = high_freq_score(hr01)
        ed = edge_density(hr01, thr=edge_thr)
        oscore = orient_structure_score(hr01, thr=edge_thr)

        # cheap score: 구조/엣지 중심 + hf 보조
        score = 0.45 * ed + 0.35 * oscore + 0.20 * hf

        if score > best_score:
            best_score = score
            best_meta = {"x": x, "y": y, "hf": hf, "ed": ed, "os": oscore}

    return best_score, best_meta


@torch.no_grad()
def preselect_top_images_by_cheap_score(
    valid_paths,
    top_m=20,
    hr_size=96,
    n_trials=15,
    edge_thr=0.08
):
    """
    valid 100장 전체에 대해 cheap score만 계산해서 top_m 이미지 path 리스트 반환
    """
    scored = []
    for p in tqdm(valid_paths, desc="cheap_rank", total=len(valid_paths)):
        s, meta = cheap_rank_one_image(
            p, hr_size=hr_size, n_trials=n_trials, edge_thr=edge_thr
        )
        if meta is None:
            continue
        scored.append({"path": p, "cheap": s, **meta})

    scored = sorted(scored, key=lambda r: r["cheap"], reverse=True)
    top = scored[:min(top_m, len(scored))]
    top_paths = [r["path"] for r in top]

    print("\n[cheap_rank top candidates]")
    for i, r in enumerate(top[:10]):  # 상위 10개 프린트
        print(f"{i+1:02d} {os.path.basename(r['path'])} cheap={r['cheap']:.4f} (x={r['x']} y={r['y']}) "
              f"ed={r['ed']:.3f} os={r['os']:.3f} hf={r['hf']:.4f}")

    return top_paths, top

In [ ]:
@torch.no_grad()
def score_subset_valid_visual(
    subset_paths,
    hr_size=96,
    scale=2.0,
    n_trials=15,
    topk=3,
    edge_thr=0.08,
    min_psnr=0.0,
    max_psnr=100.0,
    min_edge=0.0,
    min_hf=None,
):
    rows = []
    for path in tqdm(subset_paths, desc=f"visual_pick x{scale}", total=len(subset_paths)):
        best = pick_best_visual_patch(
            path,
            scale=scale,
            hr_size=hr_size,
            n_trials=n_trials,
            topk=topk,
            edge_thr=edge_thr,
            min_psnr=min_psnr,
            max_psnr=max_psnr,
            min_edge=min_edge,
            min_hf=min_hf,
        )
        if best is None:
            continue

        rows.append(best)

    rows = sorted(rows, key=lambda r: r["gap"], reverse=True)
    return rows

In [ ]:
# ============================================================
# 4) Picking logic (A gap / B improve / C representative)
# ============================================================

def pick_topk_by(rows, key, *, topk=1, top_texture=30, reverse=True):
    cand = sorted(rows, key=lambda r: r["hf"], reverse=True)[:top_texture]
    cand = sorted(cand, key=lambda r: r[key], reverse=reverse)
    return cand[:topk]

def pick_representative(rows, *, top_texture=30, key="improve"):
    cand = sorted(rows, key=lambda r: r["hf"], reverse=True)[:top_texture]
    cand = sorted(cand, key=lambda r: r[key])
    return cand[len(cand)//2]

def pick_presentation_set(rows, *, top_texture=30):
    picks = []
    picks.append(("A_gap_top",      pick_topk_by(rows, "gap",     topk=1, top_texture=top_texture)[0]))
    picks.append(("B_improve_top",  pick_topk_by(rows, "improve", topk=1, top_texture=top_texture)[0]))
    picks.append(("C_representative", pick_representative(rows, top_texture=top_texture, key="improve")))

    # 중복 제거
    seen = set()
    uniq = []
    for tag, r in picks:
        if r["path"] in seen:
            continue
        seen.add(r["path"])
        uniq.append((tag, r))
    return uniq

In [ ]:
# ============================================================
# 5) Visualization + save (같은 패치 좌표로 재현)
# ============================================================

def show_grid_pretty(lr_up, hr, outs, psnrs, title="", save_path=None):
    order = [("baseline","Base"), ("fourier5","Fourier(L=5)"), ("fourier10","Fourier(L=10)"), ("inv_area","Inv-Area")]
    fig, axes = plt.subplots(1, 2+len(order), figsize=(4*(2+len(order)), 4))

    def to_np(t):
        return t.squeeze(0).permute(1,2,0).detach().cpu().numpy().clip(0,1)

    axes[0].imshow(to_np(lr_up)); axes[0].set_title("LR (bicubic up)"); axes[0].axis("off")
    axes[1].imshow(to_np(hr));    axes[1].set_title("GT");             axes[1].axis("off")

    for i, (k, disp) in enumerate(order):
        ps = psnrs.get(k, float("nan"))
        axes[2+i].imshow(to_np(outs[k]))
        axes[2+i].set_title(f"{disp}\n(PSNR(Y)={ps:.2f})")
        axes[2+i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()

@torch.no_grad()
def run_show_save_row(row, *, scale, tag, save_dir):
    """
    row: score_all_valid_bestpatch에서 나온 row (x,y,hr_size 포함)
    같은 패치 좌표로 다시 crop해서 평가/저장
    """
    img = Image.open(row["path"]).convert("RGB")
    hr_size = row["hr_size"]
    hr_pil = crop_pil(img, row["x"], row["y"], hr_size)
    hr01 = transforms.ToTensor()(hr_pil).unsqueeze(0).to(DEVICE)

    lr_up01, outs, psnrs = eval_patch(hr01, scale=scale)

    base = os.path.splitext(os.path.basename(row["path"]))[0]
    save_path = os.path.join(save_dir, f"{base}_{tag}_hr{hr_size}_x{scale}_x{row['x']}_y{row['y']}.png")
    title = f"{os.path.basename(row['path'])} | crop=({row['x']},{row['y']}) | HR={hr_size} | scale={scale} | {tag}"
    show_grid_pretty(lr_up01, hr01, outs, psnrs, title=title, save_path=save_path)
    return save_path

In [ ]:
# ============================================================
# 6) RUN ALL
# ============================================================
SAVE_DIR = "/content/drive/MyDrive/SRNO_valid_comparisons"
os.makedirs(SAVE_DIR, exist_ok=True)

t0 = time.time()

# 1) cheap ranking으로 top20 이미지 선별
top_paths, top_meta = preselect_top_images_by_cheap_score(
    valid_files,
    top_m=20,
    hr_size=96,
    n_trials=15,
    edge_thr=0.08
)

# 2) top20에 대해서만 visual_pick 수행 (모델 forward 포함)
rows_x2 = score_subset_valid_visual(
    top_paths,
    hr_size=96,
    scale=2.0,
    n_trials=15,
    topk=3,
    edge_thr=0.08,
    min_psnr=30.0,
    max_psnr=45.0,
    min_edge=0.03,
    min_hf=None
)

print("kept:", len(rows_x2))
assert len(rows_x2) > 0, "필터가 너무 빡세서 남은 게 없음. min_psnr/n_trials/min_edge 조정!"

picked = pick_presentation_set(rows_x2, top_texture=min(20, len(rows_x2)))

print("\n📌 Picked:")
for tag, r in picked:
    print(tag, os.path.basename(r["path"]),
          f"hf={r['hf']:.4f}",
          f"gap={r['gap']:.2f}",
          f"improve={r['improve']:.2f}",
          {k: round(v,2) for k, v in r["psnr"].items()},
          f"(crop x={r['x']} y={r['y']})")

print("elapsed(sec):", round(time.time()-t0, 1))

In [ ]:
# 저장 + 출력
for tag, r in picked:
    print("\n===== SAVE:", tag, "=====")
    print("x2  ->", run_show_save_row(r, scale=2.0, tag=f"{tag}_x2", save_dir=SAVE_DIR))
    print("x3  ->", run_show_save_row(r, scale=3.0, tag=f"{tag}_x3", save_dir=SAVE_DIR))
    print("x4  ->", run_show_save_row(r, scale=4.0, tag=f"{tag}_x4", save_dir=SAVE_DIR))

## 복원용 이미지 다운로드 (공유 용도)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms

DIV2K_ROOT = "/content/DIV2K"
VALID_DIR = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR")

OUT_DIR = "/content/drive/MyDrive/valid_crops_192"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# (파일명) : (x, y)  # x,y는 "HR 원본 이미지에서의 top-left 좌표"
targets = {
    "0841.png": (1552, 220),
    # "0805.png": (1338, 187),
    # "0810.png": (1372, 579),
    "0804.png": (829, 995),
    "0850.png": (649, 788),
    "0891.png": (187, 801),
    # "0892.png": (1707, 758),
    "0866.png": (801, 696),
    # "0834.png": (327, 1089),
    # "0820.png": (947, 161),
    "0847.png": (1063, 338),

}

In [ ]:
def safe_crop_box(W, H, x, y, crop):
    x = max(0, min(x, W - crop))
    y = max(0, min(y, H - crop))
    return x, y, x + crop, y + crop

def save_hr_lr_crops(fname, x, y, hr_size=192, scale=2.0):
    path = os.path.join(VALID_DIR, fname)
    img = Image.open(path).convert("RGB")
    W, H = img.size

    # HR crop (GT)
    x1, y1, x2, y2 = safe_crop_box(W, H, x, y, hr_size)
    hr_pil = img.crop((x1, y1, x2, y2))  # 192x192

    # LR crop (model input) = bicubic downsample from HR crop
    hr = transforms.ToTensor()(hr_pil).unsqueeze(0)  # (1,3,hr,hr) in [0,1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)
    lr = F.interpolate(hr, size=(lr_size, lr_size), mode="bicubic", align_corners=False)
    lr_pil = transforms.ToPILImage()(lr.squeeze(0))

    stem = os.path.splitext(fname)[0]
    hr_out = os.path.join(OUT_DIR, f"{stem}_x{x1}_y{y1}_HR{hr_size}.png")
    lr_out = os.path.join(OUT_DIR, f"{stem}_x{x1}_y{y1}_LR{lr_size}_s{scale}.png")

    hr_pil.save(hr_out)
    lr_pil.save(lr_out)

    return hr_out, lr_out

In [ ]:
saved = []
for fname, (x, y) in targets.items():
    hr_out, lr_out = save_hr_lr_crops(fname, x, y, hr_size=192, scale=2.0)
    saved.append((fname, hr_out, lr_out))

print("Done. Saved files:")
for row in saved:
    print(row)

## 이미지 비교

In [ ]:
# ============================================================
# SRNO patch compare (Colab standalone)
# HR(192 patch) -> LR 만들기(x2/x3/x4) -> SR 복원 -> PSNR 계산 -> 저장
# Outputs: LR, GT, Baseline, Fourier5, Fourier10, LocalAgg(inv_area)
# ============================================================

# 0) Drive mount
from google.colab import drive
drive.mount("/content/drive")

import os, glob, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# ------------------------------------------------------------
# 1) 경로 설정
# ------------------------------------------------------------
HR_PATCH_DIR = "/content/drive/MyDrive/valid_crops_192"   # HR192 patch 저장된 폴더
OUT_ROOT     = "/content/drive/MyDrive/SRNO_patch_compare_outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

# best_checkpoint.pth 필요
EXP_DIRS = {
    "baseline":  "/content/drive/MyDrive/SRNO_Base_final",
    "fourier5":  "/content/drive/MyDrive/SRNO_Fourier5_final",
    "fourier10": "/content/drive/MyDrive/SRNO_Fourier10_final",
    "inv_area":  "/content/drive/MyDrive/SRNO_LocalAgg_final",
}

SCALES = [2.0, 3.0, 4.0]  # x2, x3, x4

In [ ]:
# ------------------------------------------------------------
# 2) 유틸 (norm/coords/psnr)
# ------------------------------------------------------------
def norm_01_to_m11(x): return (x - 0.5) / 0.5
def denorm_m11_to_01(x): return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)

    cell = torch.empty(1, 2, device=device)
    cell[:, 0] = 2 / w
    cell[:, 1] = 2 / h

    coords = coords.repeat(batch_size, 1, 1)
    cell   = cell.repeat(batch_size, 1)
    return coords, cell

@torch.no_grad()
def rgb_to_y(img01):
    r, g, b = img01[:,0:1], img01[:,1:2], img01[:,2:3]
    return 0.299*r + 0.587*g + 0.114*b

@torch.no_grad()
def calc_psnr_y(pred01, gt01, shave=2, eps=1e-12):
    pred_y = rgb_to_y(pred01)
    gt_y   = rgb_to_y(gt01)
    if shave > 0:
        pred_y = pred_y[..., shave:-shave, shave:-shave]
        gt_y   = gt_y[..., shave:-shave, shave:-shave]
    mse = torch.mean((pred_y - gt_y) ** 2).clamp_min(eps)
    return float((-10.0 * torch.log10(mse)).item())

def save_tensor_as_png(img01, path):
    # (1,3,H,W) or (3,H,W) in [0,1]
    if img01.dim() == 4:
        img01 = img01.squeeze(0)
    img01 = img01.clamp(0,1).detach().cpu()
    pil = transforms.ToPILImage()(img01)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pil.save(path)

# ------------------------------------------------------------
# 3) 텍스트 이미지 저장 함수
# ------------------------------------------------------------
def save_with_header(img01, out_path, header_lines, *, header_h=90, bg=(255,255,255), font_size=28):
    """
    img01: torch tensor in [0,1], (1,3,H,W) or (3,H,W)
    header_lines: ["Baseline", "PSNR 25.39"] 처럼 줄 단위 리스트
    """
    if img01.dim() == 4:
        img01 = img01.squeeze(0)
    img01 = img01.clamp(0,1).detach().cpu()
    im = transforms.ToPILImage()(img01).convert("RGB")

    W, H = im.size
    canvas = Image.new("RGB", (W, H + header_h), color=bg)
    canvas.paste(im, (0, header_h))
    draw = ImageDraw.Draw(canvas)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except:
        font = ImageFont.load_default()

    y = 10
    for line in header_lines:
        bbox = draw.textbbox((0,0), line, font=font)
        tw = bbox[2] - bbox[0]
        x = (W - tw) // 2
        draw.text((x, y), line, fill=(0,0,0), font=font)
        y += (bbox[3] - bbox[1]) + 6

    canvas.save(out_path)

# ------------------------------------------------------------
# 4) 모델 정의
# ------------------------------------------------------------
class FourierPositionalEncoding(nn.Module):
    def __init__(self, L=5):
        super().__init__()
        self.L = L
        self.register_buffer("freq_bands", 2 ** torch.linspace(0, L-1, L), persistent=False)

    def forward(self, x):  # (B,N,2)
        pe = []
        for freq in self.freq_bands.to(x.device):
            pe.append(torch.sin(x * freq * math.pi))
            pe.append(torch.cos(x * freq * math.pi))
        return torch.cat(pe, dim=-1)  # (B,N, 2*L*2)

class GalerkinAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.ln_k = nn.LayerNorm(dim)
        self.ln_v = nn.LayerNorm(dim)
        self.to_out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.to_q(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.to_k(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        k2 = k.permute(0, 2, 1, 3).reshape(B, N, C)
        v2 = v.permute(0, 2, 1, 3).reshape(B, N, C)
        k2 = self.ln_k(k2)
        v2 = self.ln_v(v2)
        k = k2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = v2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        context = torch.matmul(k.transpose(-2, -1), v) / float(N)
        out = torch.matmul(q, context)
        out = out.permute(0, 2, 1, 3).reshape(B, N, C)
        return self.to_out(out)

class ResBlock(nn.Module):
    def __init__(self, n_feats, kernel_size=3, act=nn.ReLU(True), res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
            act,
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR(nn.Module):
    def __init__(self, n_resblocks=16, n_feats=64):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, n_feats, 3, padding=1))
        m_body = [ResBlock(n_feats, 3, res_scale=0.1) for _ in range(n_resblocks)]
        m_body.append(nn.Conv2d(n_feats, n_feats, 3, padding=1))
        self.body = nn.Sequential(*m_body)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        return x + res

class SRNO_Fourier(nn.Module):
    def __init__(self, residual_scale=0.1, use_fourier=False, L=5, width=128, blocks=8, corner_agg="concat"):
        super().__init__()
        assert corner_agg in ["concat", "inv_area"]
        self.corner_agg = corner_agg

        self.encoder = EDSR(16, 64)
        self.use_fourier = use_fourier
        self.L = L
        self.pos_enc = FourierPositionalEncoding(L=L)

        corner_dim = 64 + 2 + 2
        if use_fourier:
            corner_dim += (2 * L * 2)
        in_dim = (4 * corner_dim) if (self.corner_agg == "concat") else corner_dim

        self.latent_dim = width
        self.lifting = nn.Linear(in_dim, self.latent_dim)

        self.layers = nn.ModuleList([GalerkinAttention(self.latent_dim, heads=8) for _ in range(blocks)])
        self.ffns = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(self.latent_dim),
                nn.Linear(self.latent_dim, self.latent_dim),
                nn.GELU(),
                nn.Linear(self.latent_dim, self.latent_dim),
            ) for _ in range(blocks)
        ])

        self.projection = nn.Linear(self.latent_dim, 3)
        self.residual_scale = residual_scale

    @staticmethod
    def _feat_coord_grid(Hf, Wf, device, B):
        y = torch.linspace(-1, 1, Hf, device=device, dtype=torch.float32)
        x = torch.linspace(-1, 1, Wf, device=device, dtype=torch.float32)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        grid = torch.stack([xx, yy], dim=-1).unsqueeze(0)
        return grid.repeat(B, 1, 1, 1)

    def forward(self, x, coords, cell):
        B = x.shape[0]
        coords = coords.to(dtype=torch.float32)
        cell   = cell.to(dtype=torch.float32)

        grid = coords.unsqueeze(2)

        base = F.grid_sample(x, grid, mode="bilinear", align_corners=False)
        base = base.squeeze(3).permute(0,2,1)

        feat = self.encoder(x)
        _, _, Hf, Wf = feat.shape
        feat_coord = self._feat_coord_grid(Hf, Wf, feat.device, B).permute(0,3,1,2)

        rx, ry = 1.0 / Hf, 1.0 / Wf
        eps = 1e-6
        vx_lst = [-1, 1]
        vy_lst = [-1, 1]

        rel_cell = cell.unsqueeze(1).repeat(1, coords.shape[1], 1).clone()
        rel_cell[..., 0] *= Hf
        rel_cell[..., 1] *= Wf

        preds = []
        areas = []
        for vx in vx_lst:
            for vy in vy_lst:
                coord_ = coords.clone()
                coord_[..., 0] = (coord_[..., 0] + vx * rx).clamp(-1 + eps, 1 - eps)
                coord_[..., 1] = (coord_[..., 1] + vy * ry).clamp(-1 + eps, 1 - eps)
                grid_ = coord_.unsqueeze(2)

                q_feat = F.grid_sample(feat, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)
                q_coord = F.grid_sample(feat_coord, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)

                rel_coord = coords - q_coord
                rel_coord[..., 0] *= Hf
                rel_coord[..., 1] *= Wf

                area = (rel_coord[..., 0].abs() * rel_coord[..., 1].abs())
                areas.append(area)

                parts = [q_feat, rel_coord]
                if self.use_fourier:
                    parts.append(self.pos_enc(rel_coord))
                parts.append(rel_cell)
                preds.append(torch.cat(parts, dim=-1))

        if self.corner_agg == "concat":
            z_in = torch.cat(preds, dim=-1)
        else:
            eps_w = 1e-9
            inv = [1.0 / (a + eps_w) for a in areas]
            inv_sum = (inv[0] + inv[1] + inv[2] + inv[3]).clamp_min(eps_w)
            w = [(iv / inv_sum).unsqueeze(-1) for iv in inv]
            z_in = preds[0]*w[0] + preds[1]*w[1] + preds[2]*w[2] + preds[3]*w[3]

        z = self.lifting(z_in)
        for attn, ffn in zip(self.layers, self.ffns):
            z = z + attn(z)
            z = z + ffn(z)

        residual = self.projection(z) * self.residual_scale
        return base + residual

In [ ]:
# ------------------------------------------------------------
# 5) 체크포인트 로드
# ------------------------------------------------------------
def build_model_from_config(cfg):
    return SRNO_Fourier(
        residual_scale=cfg.get("RESIDUAL_SCALE", 0.1),
        use_fourier=cfg.get("USE_FOURIER", False),
        L=cfg.get("L", 5),
        width=cfg.get("WIDTH", 128),
        blocks=cfg.get("BLOCKS", 8),
        corner_agg=cfg.get("CORNER_AGG", "concat"),
    ).to(DEVICE)

def load_best_model(exp_dir):
    ckpt_path = os.path.join(exp_dir, "best_checkpoint.pth")
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = dict(ckpt.get("config", {}))
    model = build_model_from_config(cfg)

    sd = {k.replace("module.", ""): v for k, v in ckpt["model_state_dict"].items()}
    model.load_state_dict(sd, strict=True)
    model.eval()
    return model, cfg

models = {}
model_cfgs = {}
for name, d in EXP_DIRS.items():
    models[name], model_cfgs[name] = load_best_model(d)

print("✅ models loaded:", list(models.keys()))
for k in ["baseline","fourier5","fourier10","inv_area"]:
    cfg = model_cfgs[k]
    print(f"  - {k}: USE_FOURIER={cfg.get('USE_FOURIER')} L={cfg.get('L')} "
          f"CORNER_AGG={cfg.get('CORNER_AGG')} WIDTH={cfg.get('WIDTH')} BLOCKS={cfg.get('BLOCKS')}")

In [ ]:
# ------------------------------------------------------------
# 6) HR patch -> scale별 LR -> 추론 -> 저장
# ------------------------------------------------------------
@torch.no_grad()
def eval_patch_from_hr(hr01, scale):
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    hr_n = norm_01_to_m11(hr01)
    lr_n = norm_01_to_m11(lr01)

    coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)
    cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)

    outs = {}
    psnrs = {}
    for name, m in models.items():
        pred_n = m(lr_n, coords, cell)  # (1,N,3)
        pred_n = pred_n.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()
        pred01 = denorm_m11_to_01(pred_n).clamp(0, 1)
        outs[name] = pred01
        psnrs[name] = calc_psnr_y(pred01, hr01, shave=2)
    lr_up01 = F.interpolate(lr01, size=(hr_size, hr_size),
                            mode="bicubic", align_corners=False).clamp(0, 1)

    psnrs["bicubic"] = calc_psnr_y(lr_up01, hr01, shave=2)

    return lr01.clamp(0, 1), lr_up01, outs, psnrs

order = [
    ("bicubic",   "Bicubic"),
    ("baseline",  "Baseline"),
    ("fourier5",  "Fourier5"),
    ("fourier10", "Fourier10"),
    ("inv_area",  "LocalAgg"),
]

# =========================
# 출력 옵션
# =========================
DISPLAY = True              # 화면 출력 켜기/끄기
DISPLAY_SCALES = {2.0, 3.0, 4.0}
DISPLAY_MODELS = ["baseline", "fourier10", "inv_area"]  # 화면에 띄울 모델

hr_paths = sorted(glob.glob(os.path.join(HR_PATCH_DIR, "*_HR192.png")))
print("HR patches found:", len(hr_paths))

to_tensor = transforms.ToTensor()

for hr_path in hr_paths:
    stem = os.path.splitext(os.path.basename(hr_path))[0]
    hr_pil = Image.open(hr_path).convert("RGB")
    hr01 = to_tensor(hr_pil).unsqueeze(0).to(DEVICE)

    for scale in SCALES:
        subdir = os.path.join(OUT_ROOT, stem, f"x{int(scale)}")
        os.makedirs(subdir, exist_ok=True)

        lr01, lr_up01, outs, psnrs = eval_patch_from_hr(hr01, scale)

        # 저장

        save_tensor_as_png(hr01,
                          os.path.join(subdir, "GT.png"))

        save_tensor_as_png(lr_up01,
                          os.path.join(subdir, "LR_bicubicUp_to192.png"))

        save_tensor_as_png(lr01,
                          os.path.join(subdir, f"LR_{lr01.shape[-1]}x{lr01.shape[-1]}.png"))

        lines = []
        lines.append(f"[PATCH] {stem} | scale=x{int(scale)}")
        lines.append(f"LR size: {lr01.shape[-1]} -> HR size: {hr01.shape[-1]}")
        lines.append("PSNR(Y):")

        for key, tag in order:
            p = psnrs[key]
            out_path = os.path.join(subdir, f"{tag}_PSNR{p:.2f}.png")
            if key == "bicubic":
                save_tensor_as_png(lr_up01, out_path)
            else:
                save_tensor_as_png(outs[key], out_path)
            lines.append(f"  - {tag}: {p:.4f}")

        with open(os.path.join(subdir, "metrics.txt"), "w") as f:
            f.write("\n".join(lines) + "\n")

        print(f"✅ saved: {subdir}")

        # =============================
        # 화면에 비교 출력
        # =============================
        if DISPLAY:
            fig, axes = plt.subplots(2, 4, figsize=(18, 9))

            # 첫 줄
            axes[0,0].imshow(hr01.squeeze(0).permute(1,2,0).detach().cpu())
            axes[0,0].set_title("GT")
            axes[0,0].axis("off")

            axes[0,1].imshow(lr_up01.squeeze(0).permute(1,2,0).detach().cpu())
            axes[0,1].set_title("LR (bicubic up)")
            axes[0,1].axis("off")

            axes[0,2].imshow(outs["baseline"].squeeze(0).permute(1,2,0).detach().cpu())
            axes[0,2].set_title(f"Baseline\nPSNR {psnrs['baseline']:.2f}")
            axes[0,2].axis("off")

            axes[0,3].imshow(outs["fourier5"].squeeze(0).permute(1,2,0).detach().cpu())
            axes[0,3].set_title(f"Fourier5\nPSNR {psnrs['fourier5']:.2f}")
            axes[0,3].axis("off")

            # 둘째 줄
            axes[1,0].imshow(outs["fourier10"].squeeze(0).permute(1,2,0).detach().cpu())
            axes[1,0].set_title(f"Fourier10\nPSNR {psnrs['fourier10']:.2f}")
            axes[1,0].axis("off")

            axes[1,1].imshow(outs["inv_area"].squeeze(0).permute(1,2,0).detach().cpu())
            axes[1,1].set_title(f"LocalAgg\nPSNR {psnrs['inv_area']:.2f}")
            axes[1,1].axis("off")

            axes[1,2].axis("off")
            axes[1,3].axis("off")

            plt.suptitle(f"{stem}  |  scale x{int(scale)}", fontsize=16)
            plt.tight_layout()
            plt.show()


print("ALL DONE. Output root:", OUT_ROOT)